In [1]:
# Tiny Shakespeare — Dataset Preparation

## Objective

# Prepare the tokenized corpus for language-model training.

### Steps

# 1. Load the tokenized dataset.
# 2. Split the dataset into training, validation, and test sets.
# 3. Create input/target sequences.
# 4. Understand context length.
# 5. Create batches.
# 6. Verify the next-token prediction structure.

In [1]:
import torch

In [2]:
CLEAN_PATH = "../data/processed/tiny-shakespeare_clean.txt"

with open(CLEAN_PATH, "r", encoding="utf-8") as file:
    text = file.read()

vocab = sorted(set(text))
vocab_size = len(vocab)

stoi = {character: index for index, character in enumerate(vocab)}
itos = {index: character for index, character in enumerate(vocab)}

def encode(text):
    return [stoi[character] for character in text]

def decode(tokens):
    return "".join(itos[token] for token in tokens)

encoded_text = encode(text)

data = torch.tensor(encoded_text, dtype=torch.long)

print("Vocabulary size:", vocab_size)
print("Total tokens:", len(data))

Vocabulary size: 63
Total tokens: 1114961


In [3]:
n = len(data)

train_end = int(0.90 * n)
val_end = int(0.95 * n)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

print("Total tokens :", len(data))
print("Train tokens :", len(train_data))
print("Val tokens   :", len(val_data))
print("Test tokens  :", len(test_data))

Total tokens : 1114961
Train tokens : 1003464
Val tokens   : 55748
Test tokens  : 55749


In [4]:
context_length = 128

In [5]:
x = train_data[:context_length]
y = train_data[1:context_length + 1]

print("X shape:", x.shape)
print("Y shape:", y.shape)

X shape: torch.Size([128])
Y shape: torch.Size([128])


In [6]:
print("X:")
print(decode(x.tolist()))

print("\nY:")
print(decode(y.tolist()))

X:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to 

Y:
irst Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to d


In [7]:
for i in range(10):
    input_char = decode([x[i].item()])
    target_char = decode([y[i].item()])
    
    print(
        f"Input: {repr(input_char):>4} "
        f"→ Target: {repr(target_char)}"
    )

Input:  'F' → Target: 'i'
Input:  'i' → Target: 'r'
Input:  'r' → Target: 's'
Input:  's' → Target: 't'
Input:  't' → Target: ' '
Input:  ' ' → Target: 'C'
Input:  'C' → Target: 'i'
Input:  'i' → Target: 't'
Input:  't' → Target: 'i'
Input:  'i' → Target: 'z'


In [8]:
def get_batch(data, batch_size=32, context_length=128):
    starts = torch.randint(
        0,
        len(data) - context_length - 1,
        (batch_size,)
    )

    x = torch.stack([
        data[start:start + context_length]
        for start in starts
    ])

    y = torch.stack([
        data[start + 1:start + context_length + 1]
        for start in starts
    ])

    return x, y

In [9]:
batch_size = 32

x_batch, y_batch = get_batch(
    train_data,
    batch_size=batch_size,
    context_length=context_length
)

print("X batch shape:", x_batch.shape)
print("Y batch shape:", y_batch.shape)

X batch shape: torch.Size([32, 128])
Y batch shape: torch.Size([32, 128])
